In [2]:
import numpy as np
import time

In [3]:
def generate_graph(task_amount, edge_costs):
    tasks = []
    edges_with_label = []
    edges = []

    weights_sum = np.sum(TASK_PROBABILITY_WEIGHTS)

    # generowanie zadan
    for i in range(task_amount):
        weight = np.random.randint(weights_sum)

        for j,w in enumerate(TASK_PROBABILITY_WEIGHTS):
            if weight > w:
                weight -= w
            else:
                tasks.append(TASK_TYPES[j])
                break

    # dodanie krawedzi tak by polaczyc zadania
    for j in range(1, task_amount):
        max_back = min(10,j)
        parent = np.random.randint(j-max_back, j)
        edge_value = np.random.randint(edge_costs[0], edge_costs[1]+1)
        edges.append((parent,j))
        edges_with_label.append((parent,j,edge_value))

    # dodanie dodatkowych krawedzi
    edge_probability = 0.02
    for i in range(task_amount):
        for j in range(i+1, task_amount):
            if (i,j) in edges:
                continue

            if np.random.rand() <= edge_probability:
                edge_value = np.random.randint(edge_costs[0], edge_costs[1]+1)
                edges.append((i,j))
                edges_with_label.append((i,j,edge_value))

    return tasks, edges_with_label

In [4]:
def generate_resources(resorce_amount, graph_tasks, base_price_c, base_price_d):
    resources = []

    for i in range(resorce_amount):
        
        r_type = -1
        # resource type
        if np.random.rand() < 0.3:
            r_type = 1 # common
        else:
            r_type = 0 # dedicated

        # resource params
        employ_price = 0
        if r_type == 1:
            employ_price = np.random.randint(base_price_c[0], base_price_c[1]+1)
        else:
            employ_price = np.random.randint(base_price_d[0], base_price_d[1]+1)
        
        availability = 0

        resources.append((employ_price, availability, r_type))
    return resources
        

In [5]:
def generate_task_section(tasks, edges):
    result = f"@tasks {len(tasks)}"

    edges = sorted(edges, key=lambda x: (x[0], x[1]))
    relations = {}
    for e in edges:
        parent = e[0]
        child = e[1]

        if parent in relations.keys():
            relations[parent].append((child, e[2]))
        else:
            relations[parent] = [(child, e[2])]

    for i in range(len(tasks)):
        result += f'\n{tasks[i]}_{i}'
        
        if i not in relations.keys():
            result += f' 0'
        else:
            result += f' {len(relations[i])}'

            for r in relations[i]:
                result += f' {tasks[r[0]]}_{r[0]}'
                result += f'({r[1]})'

    return result
    

In [6]:
def generate_proc_section(resources):
    result = f"@proc {len(resources)}"

    for r in sorted(resources, key=lambda x: x[0]):
        result += f'\n{r[0]} {r[1]} {r[2]}'

    return result

In [7]:
def generate_times_and_cost_sections(tasks, resources):
    times_result = "\n@times\n"
    cost_result = "@cost\n"

    for t in tasks:
        for r in sorted(resources, key=lambda x: x[0]):
            if r[2] == 0:
                if t == "DT" or t == "CDT" or t == "GT" or t == "CGT":
                    time = np.random.randint(DEDICATED_TIME[0], DEDICATED_TIME[1])
                    cost = np.random.randint(DEDICATED_COST[0], DEDICATED_COST[1])
                elif t == "UT":
                    time = -1
                    cost = -1
            elif r[2] == 1:
                if (t == "DT" or t == "CDT"):
                    time = -1
                    cost = -1
                elif t == "UT" or t == "GT" or t == "CGT":
                    time = np.random.randint(COMMON_TIME[0], COMMON_TIME[1])
                    cost = np.random.randint(COMMON_COST[0], COMMON_COST[1])
                
            times_result += f'{time} '
            cost_result += f'{cost} '
        times_result = times_result[:-1] + "\n" 
        cost_result = cost_result[:-1] + "\n"

    return times_result + cost_result
        

In [8]:
def generate_comm_section(resources_amount, channels_amount):
    result = f"@comm {channels_amount}"
    channels = []
    
    for c in range(channels_amount):
        channels.append(np.zeros(resources_amount, dtype=int))

    # minimal privileges
    for r in range(resources_amount):
        c = np.random.randint(0, channels_amount)
        channels[c][r] = 1

    # bonus privileges
    for r in range(resources_amount):
        amount = np.random.randint(0, len(channels))
        channels_indexes = np.random.choice(len(channels), amount, replace=False)
        
        for i in channels_indexes:
            channels[i][r] = 1


    for i,c in enumerate(channels):
        cost = np.random.randint(CHANNEL_COST[0], CHANNEL_COST[1])
        bandwidth = np.random.randint(CHANNEL_BANDWIDTH[0], CHANNEL_BANDWIDTH[1])

        result += f"\nCH_{i} {cost} {bandwidth}"
        for privilege in c:
            result += f" {privilege}"

    return result

In [9]:
def generate_file():
    tasks, edges = generate_graph(TASK_AMOUNT, EDGE_LABEL)
    resources = generate_resources(RESOURCE_AMOUNT, tasks, BASE_PRICE_COMMON, BASE_PRICE_DEDICATED)

    result = ""
    result += generate_task_section(tasks, edges) + "\n"
    result += generate_proc_section(resources)
    result += generate_times_and_cost_sections(tasks, resources)
    result += generate_comm_section(RESOURCE_AMOUNT, CHANNEL_AMOUNT)

    return result

In [10]:
def save_file(data):
    timestamp = int(time.time())
    filename = f'GENERATED_GRAPH_{timestamp}.txt'

    with open(filename, "w") as file:
        file.write(data)

In [21]:
# graph data
TASK_TYPES = ["UT","GT","DT","CGT","CDT"]
BASE_PRICE_DEDICATED = [1000,1001]

TASK_AMOUNT = 10
TASK_PROBABILITY_WEIGHTS = [2,5,2,12,6]

RESOURCE_AMOUNT = 10
BASE_PRICE_COMMON = [20,40]
DEDICATED_COST = [60,100]
DEDICATED_TIME = [20,40]
COMMON_COST = [15,30]
COMMON_TIME = [50,90]

EDGE_LABEL = [20,40]
CHANNEL_COST = [30,50]
CHANNEL_AMOUNT = 3
CHANNEL_BANDWIDTH = [10,30]

data = generate_file()
save_file(data) 
print(data)

@tasks 10
CGT_0 3 DT_1(37) GT_3(33) CDT_7(30)
DT_1 3 CDT_2(32) CDT_6(21) DT_8(37)
CDT_2 2 GT_3(27) GT_9(33)
GT_3 2 GT_4(33) DT_5(21)
GT_4 0
DT_5 1 GT_9(31)
CDT_6 0
CDT_7 0
DT_8 0
GT_9 0
@proc 10
21 0 1
26 0 1
31 0 1
32 0 1
40 0 1
1000 0 0
1000 0 0
1001 0 0
1001 0 0
1001 0 0
@times
84 51 81 50 80 22 28 29 38 27
-1 -1 -1 -1 -1 31 23 24 30 36
-1 -1 -1 -1 -1 29 21 29 39 32
57 72 63 81 74 29 32 35 34 33
73 60 80 57 56 28 23 21 32 34
-1 -1 -1 -1 -1 26 37 26 36 24
-1 -1 -1 -1 -1 31 23 32 35 34
-1 -1 -1 -1 -1 25 34 20 26 32
-1 -1 -1 -1 -1 37 20 31 34 30
56 79 71 66 56 21 36 22 37 21
@cost
29 19 15 26 18 79 73 80 93 68
-1 -1 -1 -1 -1 83 66 99 85 66
-1 -1 -1 -1 -1 94 72 67 97 89
21 29 17 15 16 75 76 82 82 80
23 20 18 22 15 69 82 87 68 71
-1 -1 -1 -1 -1 70 99 92 86 88
-1 -1 -1 -1 -1 60 78 71 90 81
-1 -1 -1 -1 -1 67 83 86 60 78
-1 -1 -1 -1 -1 76 79 77 67 79
24 15 25 29 27 96 78 99 74 99
@comm 3
CH_0 39 22 1 1 1 0 0 1 1 1 1 0
CH_1 43 12 1 0 1 0 0 1 1 0 1 1
CH_2 31 11 0 1 0 1 1 0 1 1 1 1
